# PicoCal — Training plan

Design-only plan for the first transformer training, built on notebook 02 (training-ready dataset) and notebook 03 (detector regions). It fixes the dataset, the metric, the baselines, and an experiment matrix triaged by cost vs paper value. No training has run yet. The full version is in `reports/experiment-plan.md`.

## Fixed dataset definition

Unless an experiment varies one of these:
- Cut `sig_flux_prod_vertex_z < 100` before building the dataset — keeps 36,852 / 199,538 = 18.5% (prompt photons; removes material conversions).
- Drop duplicate-cluster groups — after the vertex cut this removes 0 additional clusters, so the two cuts are nearly equivalent here.
- Seed = the most energetic cell we compute, not the stored `seed_cell_x,y`.
- Window = 3x3 around the seed — caps tokens at ~24 (median 9) vs up to 528; the RAM fix for the <12 GB machine.
- Timing features OUT for the first stage.
- Target = `log(E_true)`, `E_true = sig_flux_eTot`.
- Region = R3 (60 mm) — most surviving data and a stable ranking across the 20/40/60/80/100-file convergence check.

## Primary metric — energy resolution

For each cluster, the relative residual

    r = (E_reco - E_true) / E_true

Report the width of `r` two ways (robust to tails):
- `sigma_eff` = half-width of the smallest interval containing 68.3% of `r`,
- IQR width = (Q75 - Q25) / 1.349.

Also report the median of `r` as the bias. Slice both **per true-energy bin** and **per region** (resolution vs E is the physics curve, roughly a/sqrt(E) (+) b). The model predicts `log(E)`; convert back to linear E before computing `r`.

## Baselines to beat

On the same kept R3 test set and metric. Two tiers:

Analytic (is ML better than equations?):
- **B0 sum-of-cells** — predicted E = sum of cell energies in the 3x3 window.
- **B1 seed-cell energy** — energy of the single most energetic cell (floor).
- **B2 `total_energy` branch** — LHCb's own per-cluster number (the rule-based target to beat).
- **calibrated sum** — affine fit of `log(E_true)` on `log(sum-of-cells)`; corrects mean scale only.

Learned (does the transformer beat cheap ML on hand-made features?):
- **B3 BDT** — sklearn `HistGradientBoostingRegressor` (CPU, seconds) on ~6 aggregate features per cluster: `sum(E)`, front/back ratio, n cells, seed energy, lateral RMS about the seed, region index. The fair competitor; if the transformer can't beat B3, the set/attention structure isn't justified.

Ladder: sum -> calibrated sum -> BDT -> Deep Sets -> transformer; each rung isolates one gain.

## Staged strategy

Each step stays a single, homogeneous problem.

```mermaid
flowchart TB
  S1["Step 1 - one region R3, 3x3 window, no timing, target = true energy"] --> S2["Step 2 - all-regions vs per-region; grow to 5x5"]
  S2 --> S3["Step 3 - add cleaned timing features"]
  S3 --> S4["Step 4 - compare seed vs reconstructed cluster position"]
  S4 --> S5["Step 5 - relax cuts / scale to all regions (CERN)"]
```


## Experiment matrix

| ID | What varies | Hypothesis | Priority | Reasoning |
|----|-------------|------------|----------|-----------|
| E0 | Analytic baselines B0/B1/B2 + calibrated sum | sum-of-cells decent but biased low | **MUST-RUN** | Defines the bar; near-zero cost |
| E0b | B3 BDT on ~6 aggregate features | cheap ML beats analytic | **MUST-RUN** | The fair "is deep learning worth it?" control; seconds on CPU |
| E1 | Transformer baseline (R3, 3x3, seed, log E, no timing) | beats E0, esp. tails | **MUST-RUN** | The headline model; fits <12 GB |
| E2 | POS_REF: seed vs cluster position | seed-relative coords tighter | **MUST-RUN** | Mentor-requested; flag exists; cheap |
| E4 | All-regions model vs per-region | one model across 8x geometries is worse | **MUST-RUN** | Tests the core "one region first" claim |
| E3 | R3 vs each of R0,R1,R2,R4 | inner regions resolve better; R4 starved | NICE-TO-HAVE | Good figure; R4 (2,479) may be too data-poor |
| E5 | Window 3x3 vs 5x5 | 5x5 lowers bias at higher RAM | NICE-TO-HAVE | RAM-bound; defer to CERN |
| E6 | Cuts: filtered vs all-data (fair variant only) | naive all-data is ill-posed | NICE-TO-HAVE | See validity note |
| E7 | Timing off vs on | little gain at first stage | NICE-TO-HAVE | Later stage |
| E8 | Target log(E) vs linear E | log(E) trains more stably | NICE-TO-HAVE | Cheap methodology footnote |
| E9 | Full clusters, no window | upper bound if RAM unlimited | **SKIP** | Exactly the config that OOMs |
| E10 | Architecture/HP sweep | bigger marginally better | **SKIP (for now)** | Low paper value before axes settled |
| E11 | Deep Sets (no attention) vs transformer | attention adds resolution over plain pooling | **MUST-RUN** | Defends the architecture choice; gives attention its own credit |

## Validity note for the all-data comparison (E6)

A naive "train on all 199,538 entries" run is **ill-posed**: 66.7% of entries are in duplicate-cluster groups (one cluster maps to several true photons with different targets — identical inputs, two answers), and it includes material-conversion photons the vertex cut removes. Its loss is not comparable to the filtered loss.

The fair version of "effect of the cuts" is either:
- (a) train on filtered data and report the coverage cost (how much was discarded), or
- (b) keep all clusters but pick one well-defined target per duplicate group (e.g. the highest-energy photon) so the mapping is a function, then compare.

Only (a)/(b) belong in the paper; the naive all-data loss is at most a cautionary illustration.

## Recommended order

1. **E0 + E0b** — analytic baselines then the B3 BDT (seconds, CPU); first, so E1 is interpretable.
2. **E1** — transformer baseline (R3, 3x3, seed, log E, no timing).
3. **E2** — POS_REF seed vs cluster.
4. **E11** — Deep Sets vs transformer (defends the attention choice).
5. **E4** — all-regions vs per-region.
6. **E8** then **E6 (fair)** — cheap local methodology checks.
7. **E3** — per-region resolution figure (R4 indicative only).
8. **E5** and any full-region work — **defer to CERN** (RAM-bound).
9. **E7** — timing, once the energy baseline is solid.

## Full plan

The complete matrix with metric definitions, cost estimates, and the skip rationale is in `reports/experiment-plan.md`.

In [1]:
{
    "fixed": {"region": "R3 (60mm)", "window": "3x3", "timing": False,
              "target": "log(E_true)", "pos_ref": "seed", "cut": "vertex_z<100"},
    "metric": "width of r=(E_reco-E_true)/E_true: sigma_eff (68.3%) and IQR/1.349; per E-bin and region",
    "must_run": ["E0", "E0b", "E1", "E2", "E11", "E4"],
    "baselines": ["B0 sum", "B1 seed", "B2 total_energy", "calibrated sum", "B3 BDT"],
    "skip": ["E9", "E10", "E6 (naive all-data)"],
    "defer_to_cern": ["E5", "full-region runs"],
    "details": "reports/experiment-plan.md",
}

{'fixed': {'region': 'R3 (60mm)',
  'window': '3x3',
  'timing': False,
  'target': 'log(E_true)',
  'pos_ref': 'seed',
  'cut': 'vertex_z<100'},
 'metric': 'width of r=(E_reco-E_true)/E_true: sigma_eff (68.3%) and IQR/1.349; per E-bin and region',
 'must_run': ['E0', 'E0b', 'E1', 'E2', 'E11', 'E4'],
 'baselines': ['B0 sum',
  'B1 seed',
  'B2 total_energy',
  'calibrated sum',
  'B3 BDT'],
 'skip': ['E9', 'E10', 'E6 (naive all-data)'],
 'defer_to_cern': ['E5', 'full-region runs'],
 'details': 'reports/experiment-plan.md'}